# Memory in LangChain

## Overview

**Memory** is a crucial component in LangChain that allows chains and agents to remember information from previous interactions.

### Why Memory is Important?

- **Context Preservation**: Maintain conversation history across multiple interactions
- **Personalization**: Remember user preferences and information
- **Continuity**: Enable natural, flowing conversations
- **State Management**: Track conversation state for complex workflows

### Memory Architecture Diagram

```
┌─────────────┐
│   User      │
│   Input     │
└──────┬──────┘
       │
       ▼
┌─────────────────┐
│   Memory Store  │ ◄─── Stores conversation history
│  (Session ID)   │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Prompt +       │
│  Chat History   │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│      LLM        │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│    Response     │
└─────────────────┘
         │
         ▼
┌─────────────────┐
│  Update Memory  │
└─────────────────┘
```

**Flow**: User Input → Memory Store → Combine with History → LLM → Response → Update Memory

## Types of Memory

### 1. ConversationBufferMemory

Stores the entire conversation history as a list of messages.

**Flow Diagram:**

```
┌─────────────────────────────────────────┐
│      ConversationBufferMemory           │
│                                          │
│  Message 1: User: "Hello"              │
│  Message 2: AI: "Hi there!"            │
│  Message 3: User: "My name is John"    │
│  Message 4: AI: "Nice to meet you..." │
│  Message 5: User: "What's my name?"    │
│  Message 6: AI: "Your name is John"   │
│  ... (stores ALL messages)              │
└─────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────┐
│         All Messages → LLM              │
│    (Complete Context Available)         │
└─────────────────────────────────────────┘
```

---

### 2. ConversationSummaryMemory

Stores a summary of the conversation history instead of all messages.

**Flow Diagram:**

```
┌─────────────────────────────────────────┐
│      Original Conversation              │
│                                          │
│  Msg 1: "Hello"                        │
│  Msg 2: "Hi there!"                    │
│  Msg 3: "My name is John"              │
│  Msg 4: "Nice to meet you..."         │
│  Msg 5: "I like Python"                │
│  ... (many more messages)               │
└──────────────┬──────────────────────────┘
               │
               ▼ Summarize
┌─────────────────────────────────────────┐
│    ConversationSummaryMemory            │
│                                          │
│  Summary: "User introduced themselves   │
│           as John. They like Python     │
│           programming..."               │
│  (Only key information preserved)        │
└──────────────┬──────────────────────────┘
               │
               ▼
┌─────────────────────────────────────────┐
│    Summary + New Message → LLM          │
│    (Efficient, preserves key info)      │
└─────────────────────────────────────────┘
```

---

### 3. ConversationBufferWindowMemory

Stores only the last N messages from the conversation.

**Flow Diagram:**

```
┌─────────────────────────────────────────┐
│      Full Conversation History          │
│                                          │
│  [Old] Msg 1: "Hello"                  │
│  [Old] Msg 2: "Hi there!"              │
│  [Old] Msg 3: "My name is John"        │
│  [Old] Msg 4: "Nice to meet you..."    │
│  ─────────────────────────────────────  │
│  [Keep] Msg 5: "I like Python"         │
│  [Keep] Msg 6: "That's great!"         │
│  [Keep] Msg 7: "What's my name?"       │
│  [Keep] Msg 8: "Your name is John"     │
│  [Keep] Msg 9: "Thanks!"              │
└──────────────┬──────────────────────────┘
               │
               ▼ Keep only last N=5
┌─────────────────────────────────────────┐
│  ConversationBufferWindowMemory (k=5)    │
│                                          │
│  ✓ Msg 5: "I like Python"              │
│  ✓ Msg 6: "That's great!"              │
│  ✓ Msg 7: "What's my name?"            │
│  ✓ Msg 8: "Your name is John"          │
│  ✓ Msg 9: "Thanks!"                    │
│  ✗ Older messages discarded            │
└──────────────┬──────────────────────────┘
               │
               ▼
┌─────────────────────────────────────────┐
│    Last N Messages → LLM                │
│    (Fixed size, recent context only)    │
└─────────────────────────────────────────┘
```

## Setup

First, let's set up the necessary imports and LLM configuration.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from config import settings

# Initialize LLM
llm = ChatOpenAI(
    model=settings.LLM_CHAT_MODEL,
    api_key=settings.LLM_API_KEY,
    base_url=settings.LLM_BASE_URL,
    temperature=0.7
)

## Basic Example: Using Memory with Chains

In [3]:
# Simple example: Remember user's name
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory # wrap chain with memory
from langchain_community.chat_message_histories import ChatMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý AI thân thiện."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

# Simple in-memory store
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)
response1 = chain_with_memory.invoke(
    {"input": "Tên tôi là Hùng"},
    config={"configurable": {"session_id": "1"}}
)
print(f"User: Tên tôi là Hùng")
print(f"AI: {response1[:100]}...\n")

response2 = chain_with_memory.invoke(
    {"input": "Tên tôi là gì?"},
    config={"configurable": {"session_id": "1"}}
)
print(f"User: Tên tôi là gì?")
print(f"AI: {response2[:100]}...")

User: Tên tôi là Hùng
AI: Chào Hùng! Rất vui được làm quen với bạn. 😊

Tôi là trợ lý AI của bạn. Hôm nay bạn thế nào? Tôi có t...

User: Tên tôi là gì?
AI: Tên bạn là **Hùng** ạ! 😊 Bạn vừa mới giới thiệu với mình xong mà.

Hôm nay Hùng có chuyện gì vui muố...


## Example 2: ConversationSummaryMemory

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tóm tắt ngắn gọn cuộc hội thoại sau, giữ lại thông tin quan trọng."),
    ("human", "{history}")
])

summary_chain = summary_prompt | llm | StrOutputParser()
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


def summarize_if_needed(history: ChatMessageHistory):
    if len(history.messages) > 6:
        text = "\n".join(m.content for m in history.messages)
        summary = summary_chain.invoke({"history": text})
        history.clear()
        history.add_ai_message("Summary so far: " + summary)
from langchain_core.runnables import RunnableWithMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý AI thân thiện."),
    ("human", "{input}")
])

chat_chain = chat_prompt | llm | StrOutputParser()

chat_with_memory = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input"
)
response1 = chat_with_memory.invoke(
    {"input": "Tên tôi là Hùng"},
    config={"configurable": {"session_id": "1"}}
)
print(f"User: Tên tôi là Hùng")
print(f"AI: {response1[:100]}...")
response2 = chat_with_memory.invoke(
    {"input": "Tên tôi là gì?"},
    config={"configurable": {"session_id": "1"}}
)
print(f"User: Tên tôi là gì?")
print(f"AI: {response2[:100]}...")


User: Tên tôi là Hùng
AI: Chào Hùng! Rất vui được làm quen với bạn. Tôi là trợ lý AI thân thiện. 

Hôm nay Hùng có cần tôi giú...
User: Tên tôi là gì?
AI: Tên bạn là **Hùng** ạ. Tôi vẫn nhớ mà! 😊

Hôm nay Hùng có chuyện gì vui muốn kể cho tôi nghe không?...
